# Temat 2: Detekcja twarzy – Viola-Jones (2001)

## Notebook demonstracyjny

Projekt realizowany w ramach przedmiotu ADOM 26L.

Artykuł: P. Viola, M. Jones, *"Rapid Object Detection using a Boosted Cascade of Simple Features"*, CVPR 2001.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from IPython.display import display, HTML

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")

## 1. Kluczowe koncepcje algorytmu Viola-Jones

### 1.1 Cechy Haara (Haar-like features)

Algorytm wykorzystuje proste cechy prostokątne, które mierzą różnice jasności między sąsiednimi regionami obrazu.

In [ ]:
def visualize_haar_features():
    """Wizualizacja typów cech Haara."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    feature_types = [
        ("Dwuprostokątna\n(pionowa)", [(0, 0, 50, 100, 'white'), (50, 0, 50, 100, 'gray')]),
        ("Dwuprostokątna\n(pozioma)", [(0, 0, 100, 50, 'white'), (0, 50, 100, 50, 'gray')]),
        ("Trzyprostokątna", [(0, 0, 33, 100, 'white'), (33, 0, 34, 100, 'gray'), (67, 0, 33, 100, 'white')]),
        ("Czteroprostokątna", [(0, 0, 50, 50, 'white'), (50, 0, 50, 50, 'gray'), 
                               (0, 50, 50, 50, 'gray'), (50, 50, 50, 50, 'white')]),
    ]
    
    for ax, (title, rects) in zip(axes, feature_types):
        img = np.ones((100, 100, 3), dtype=np.uint8) * 200
        for x, y, w, h, color in rects:
            c = (255, 255, 255) if color == 'white' else (100, 100, 100)
            cv2.rectangle(img, (x, y), (x+w, y+h), c, -1)
            cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 0), 1)
        ax.imshow(img)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    
    plt.suptitle('Typy cech Haara (Haar-like features)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_haar_features()

### 1.2 Obraz całkowy (Integral Image)

Obraz całkowy umożliwia obliczenie sumy pikseli w dowolnym prostokącie w **stałym czasie** (4 odwołania do tablicy).

$$ii(x, y) = \sum_{x' \leq x, y' \leq y} i(x', y')$$

In [ ]:
def demonstrate_integral_image():
    """Demonstracja obrazu całkowego."""
    # Przykładowy mały obraz
    img = np.array([
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]
    ], dtype=np.float64)
    
    # Obraz całkowy
    integral = cv2.integral(img.astype(np.uint8))
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Oryginalny
    im0 = axes[0].imshow(img, cmap='Blues', vmin=0)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            axes[0].text(j, i, f'{int(img[i,j])}', ha='center', va='center', fontsize=14)
    axes[0].set_title('Obraz oryginalny i(x,y)', fontsize=12)
    plt.colorbar(im0, ax=axes[0], shrink=0.8)
    
    # Całkowy
    im1 = axes[1].imshow(integral, cmap='Oranges', vmin=0)
    for i in range(integral.shape[0]):
        for j in range(integral.shape[1]):
            axes[1].text(j, i, f'{int(integral[i,j])}', ha='center', va='center', fontsize=11)
    axes[1].set_title('Obraz całkowy ii(x,y)', fontsize=12)
    plt.colorbar(im1, ax=axes[1], shrink=0.8)
    
    plt.tight_layout()
    plt.show()
    
    # Demonstracja obliczania sumy prostokąta
    print("Suma pikseli w prostokącie (1,1)-(2,2):")
    rect_sum = integral[3, 3] - integral[1, 3] - integral[3, 1] + integral[1, 1]
    direct_sum = img[1:3, 1:3].sum()
    print(f"  Przez obraz całkowy: {int(rect_sum)} (4 odwołania)")
    print(f"  Suma bezpośrednia:   {int(direct_sum)} (weryfikacja)")

demonstrate_integral_image()

### 1.3 AdaBoost – selekcja cech

AdaBoost wybiera spośród >180 000 możliwych cech Haara te najbardziej informatywne.
Każda runda boostingu wybiera **jedną cechę** minimalizującą ważony błąd klasyfikacji.

In [ ]:
def visualize_adaboost_concept():
    """Wizualizacja koncepcji AdaBoost w kontekście Viola-Jones."""
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Symulacja selekcji cech w kolejnych rundach
    rounds = range(1, 21)
    error_rates = [0.35 - 0.015 * r + np.random.normal(0, 0.01) for r in rounds]
    error_rates = [max(0.05, min(0.45, e)) for e in error_rates]
    
    ax.bar(rounds, error_rates, color='#2196F3', alpha=0.7, edgecolor='#1565C0')
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Losowe zgadywanie')
    ax.set_xlabel('Runda boostingu (numer wybranej cechy)')
    ax.set_ylabel('Błąd ważony wybranej cechy')
    ax.set_title('AdaBoost: Błąd najlepszej cechy w kolejnych rundach\n'
                 '(cechy z wczesnych rund: 0.1-0.3, późniejsze: 0.4-0.5)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

visualize_adaboost_concept()

### 1.4 Kaskada klasyfikatorów

Kaskada pozwala szybko odrzucać regiony niezawierające twarzy.
Pełna kaskada ma **38 etapów** i ponad **6000 cech**, ale średnio ewaluuje tylko **~10 cech** na podokno.

In [ ]:
def visualize_cascade():
    """Wizualizacja działania kaskady."""
    stages = ['Etap 1\n(1 cecha)', 'Etap 2\n(10 cech)', 'Etap 3\n(25 cech)', 
              'Etap 4\n(25 cech)', 'Etap 5\n(50 cech)', '...', 'Etap 38\n(~200 cech)']
    
    # Symulacja: ile podokien przechodzi każdy etap (z ~75 mln)
    total_windows = 75_000_000
    remaining = [total_windows]
    rejection_rates = [0.5, 0.8, 0.9, 0.95, 0.97, 0.99, 0.999]
    
    for rate in rejection_rates:
        remaining.append(int(remaining[-1] * (1 - rate)))
    
    fig, ax = plt.subplots(figsize=(12, 5))
    
    labels = ['Wejście'] + stages
    vals = remaining[:len(labels)]
    
    colors = ['#F44336'] + ['#4CAF50' if i < 5 else '#FF9800' for i in range(len(stages))]
    
    ax.barh(range(len(labels)), vals, color=colors, alpha=0.8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel('Liczba podokien do zbadania')
    ax.set_title('Kaskada klasyfikatorów: szybka eliminacja negatywnych podokien')
    ax.set_xscale('log')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    for i, v in enumerate(vals):
        ax.text(v * 1.5, i, f'{v:,.0f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

visualize_cascade()

## 2. Detekcja twarzy z OpenCV

### 2.1 Załadowanie klasyfikatora kaskadowego

In [ ]:
# Załaduj pretrenowany klasyfikator kaskadowy
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)

print(f"Klasyfikator załadowany: {cascade_path}")
print(f"Pusty: {face_cascade.empty()}")

### 2.2 Detekcja na obrazie testowym

In [ ]:
def create_test_face_image(n_faces=3, w=640, h=480):
    """Generuj obraz testowy z syntetycznymi twarzami."""
    np.random.seed(42)
    img = np.random.randint(100, 180, (h, w, 3), dtype=np.uint8)
    
    gt_boxes = []
    for i in range(n_faces):
        cx = 100 + i * 200
        cy = 200 + (i % 2) * 60
        fw, fh = 90, 110
        
        skin = (200, 180, 160)
        cv2.ellipse(img, (cx, cy), (fw//2, fh//2), 0, 0, 360, skin, -1)
        
        ey = cy - fh // 5
        cv2.ellipse(img, (cx - fw//5, ey), (fw//10, fh//14), 0, 0, 360, (30, 30, 30), -1)
        cv2.ellipse(img, (cx + fw//5, ey), (fw//10, fh//14), 0, 0, 360, (30, 30, 30), -1)
        
        cv2.ellipse(img, (cx, cy + fh//8), (fw//12, fh//12), 0, 0, 360,
                    (skin[0]+10, skin[1]+10, skin[2]+10), -1)
        
        gt_boxes.append((cx - fw//2, cy - fh//2, fw, fh))
    
    return img, gt_boxes


def detect_and_show(img, cascade, scale_factor=1.3, min_neighbors=5, title='Detekcja'):
    """Wykryj twarze i wyświetl wynik."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    
    start = time.perf_counter()
    faces = cascade.detectMultiScale(
        gray, scaleFactor=scale_factor, minNeighbors=min_neighbors,
        minSize=(30, 30), flags=cv2.CASCADE_SCALE_IMAGE
    )
    elapsed = time.perf_counter() - start
    
    result = cv2.cvtColor(img.copy(), cv2.COLOR_BGR2RGB)
    for (x, y, w, h) in faces:
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 0), 2)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(result)
    plt.title(f"{title}\nscaleFactor={scale_factor}, minNeighbors={min_neighbors}\n"
              f"Wykryte twarze: {len(faces)}, Czas: {elapsed*1000:.1f} ms")
    plt.axis('off')
    plt.show()
    
    return faces, elapsed


img, gt = create_test_face_image()
faces, _ = detect_and_show(img, face_cascade)

## 3. Wpływ parametrów

### 3.1 scaleFactor

Kontroluje krok skalowania przy przeszukiwaniu piramidy obrazów.

In [ ]:
scale_factors = [1.01, 1.05, 1.1, 1.2, 1.3, 1.5]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, sf in zip(axes.flat, scale_factors):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    
    start = time.perf_counter()
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=sf, minNeighbors=5, minSize=(30, 30)
    )
    elapsed = time.perf_counter() - start
    
    result = cv2.cvtColor(img.copy(), cv2.COLOR_BGR2RGB)
    for (x, y, w, h) in faces:
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 0), 2)
    
    ax.imshow(result)
    ax.set_title(f'scaleFactor={sf}\nTwarze: {len(faces)}, Czas: {elapsed*1000:.1f}ms', fontsize=10)
    ax.axis('off')

plt.suptitle('Wpływ scaleFactor na detekcję (minNeighbors=5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.2 minNeighbors

Kontroluje minimalną liczbę nakładających się detekcji wymaganą do zaakceptowania wyniku.

In [ ]:
min_neighbors_list = [0, 1, 2, 3, 5, 10]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, mn in zip(axes.flat, min_neighbors_list):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.3, minNeighbors=mn, minSize=(30, 30)
    )
    
    result = cv2.cvtColor(img.copy(), cv2.COLOR_BGR2RGB)
    for (x, y, w, h) in faces:
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 0), 2)
    
    ax.imshow(result)
    ax.set_title(f'minNeighbors={mn}\nTwarze: {len(faces)}', fontsize=10)
    ax.axis('off')

plt.suptitle('Wpływ minNeighbors na detekcję (scaleFactor=1.3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Wpływ oświetlenia

In [ ]:
def adjust_gamma(image, gamma):
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype('uint8')
    return cv2.LUT(image, table)


gammas = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, gamma in zip(axes.flat, gammas):
    dark = adjust_gamma(img, gamma)
    gray = cv2.cvtColor(dark, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30)
    )
    
    result = cv2.cvtColor(dark.copy(), cv2.COLOR_BGR2RGB)
    for (x, y, w, h) in faces:
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 0), 2)
    
    label = 'jasny' if gamma < 1 else ('normalny' if gamma == 1 else 'ciemny')
    ax.imshow(result)
    ax.set_title(f'γ={gamma} ({label})\nTwarze: {len(faces)}', fontsize=10)
    ax.axis('off')

plt.suptitle('Wpływ oświetlenia na detekcję (z equalizacją histogramu)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Podsumowanie

### Kluczowe wnioski z artykułu Viola-Jones (2001):

1. **Trzy główne kontrybucje**: Obraz całkowy, AdaBoost do selekcji cech, kaskada klasyfikatorów
2. **Szybkość**: 15 FPS na 384×288 (Pentium III, 700 MHz) – przełom w 2001 roku
3. **Kaskada**: 38 etapów, 6061 cech, ale średnio ~10 ewaluacji na podokno
4. **Wyniki**: 93.9% detection rate przy 167 fałszywych detekcjach (MIT+CMU test set)
5. **Ograniczenia**: Tylko twarze frontalne, wrażliwość na oświetlenie i okluzję